In [2]:
! pip list

Package                 Version
----------------------- ----------
annotated-doc           0.0.7
annotated-types         0.8.0
anyio                   4.16.0
appnope                 1.0.0
asttokens               3.0.2
certifi                 2026.11.20
charset-normalizer      3.5.3
click                   8.5.1
cloudpickle             3.1.3
comm                    0.2.3
debugpy                 1.8.22
executing               2.3.0
faiss-cpu               1.16.0
filelock                4.3.0
fsspec                  2026.9.0
h11                     0.16.0
hf-xet                  1.8.0
httpcore2               2.16.2
httpx2                  2.16.2
huggingface_hub         2.4.0
idna                    3.20
ipykernel               7.4.0
ipython                 9.18.1
ipython_pygments_lexers 1.1.1
jedi                    0.20.1
Jinja2                  3.1.6
joblib                  1.6.0
jupyter_client          8.10.0
jupyter_core            5.9.1
markdown-it-py          4.3.0
MarkupSafe       

In [3]:
import sys
import numpy
import requests
import sentence_transformers
import faiss

print("Python:", sys.version.split()[0])
print("numpy:", numpy.__version__)
print("requests:", requests.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("faiss:", faiss.__version__)


/Users/diman/git_projects/praktikum-ai-engineer/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 11:20:13.666000 15463 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Python: 3.15.2
numpy: 2.5.5
requests: 2.34.2
sentence-transformers: 6.3.0
faiss: 1.16.0


In [4]:
import requests

# URL локального сервера Ollama
OLLAMA_URL = "http://localhost:11434"

# Отправляем GET-запрос на эндпоинт /api/tags,
# который возвращает список локально доступных моделей
response = requests.get(f"{OLLAMA_URL}/api/tags")

# Достаём из JSON-ответа список моделей
data = response.json()
models = data.get("models", [])

# Выводим количество найденных моделей и их имена
print(f"Найдено моделей: {len(models)}")
for model in models:
    print(f" - {model['name']}")


Найдено моделей: 2
 - huihui_ai/Qwen3.8-abliterated:latest
 - qwen2.5:1.5b


# Чанки

In [5]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [6]:
from pathlib import Path

DOCS_DIR = Path("docs")

documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}")

Загружено документов: 54


In [7]:
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

Всего чанков: 500
Средняя длина чанка: 558 символов


In [8]:
for chunk in chunks[:3]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

--- из docs/README.md (669 символов) ---
# Documentation

### Getting Started
* [Quickstart](https://docs.ollama.com/quickstart)
* [Examples](./examples.md)
* [Importing models](https://docs.ollama.com/import)
* [MacOS Documentation](https:/ ...

--- из docs/README.md (147 символов) ---
* [Troubleshooting Guide](https://docs.ollama.com/troubleshooting)
* [FAQ](https://docs.ollama.com/faq#faq)
* [Development guide](./development.md) ...

--- из docs/api/anthropic-compatibility.mdx (230 символов) ---
---
title: Anthropic compatibility
---

Ollama provides compatibility with the [Anthropic Messages API](https://docs.anthropic.com/en/api/messages) to help connect existing applications to Ollama, inc ...



In [9]:
from pathlib import Path
from collections import Counter

DOCS_DIR = Path("docs")

# Загрузка и чанкинг - это вы уже видели в теории
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

# 1. Сколько разных файлов дали чанки?
unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

# 2. Сколько чанков пришло из каждого файла?
#    Подсказка: соберите Counter по c["source"] для всех чанков
source_counts = Counter(c["source"] for c in chunks)

# 3. Топ-3 файла по числу чанков
top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков")

Всего чанков: 500
Файлов с чанками: 54
Топ-3 файла по числу чанков:
  docs/api.md: 76 чанков
  docs/faq.mdx: 37 чанков
  docs/capabilities/tool-calling.mdx: 33 чанков


In [19]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")

for max_chars in [200, 800, 3000]:
    chunks_sample = chunk_text(sample_text, max_chars=max_chars)
    # Ваш код здесь: посчитайте количество чанков и среднюю длину
    # Выведите результат в формате: max_chars=200 -> чанков: X, средняя длина: Y
    print(f"max_chars={max_chars} -> чанков: {len(chunks_sample)}, средняя длина: {sum(len(c) for c in chunks_sample) // len(chunks_sample)}")
    if max_chars == 3000:
        print(chunks_sample[5])

max_chars=200 -> чанков: 170, средняя длина: 321
max_chars=800 -> чанков: 76, средняя длина: 736
max_chars=3000 -> чанков: 32, средняя длина: 1751
```shell
curl -X POST http://localhost:11434/api/generate -H "Content-Type: application/json" -d '{
  "model": "llama3.1:8b",
  "prompt": "Ollama is 22 years old and is busy saving the world. Respond using JSON",
  "stream": false,
  "format": {
    "type": "object",
    "properties": {
      "age": {
        "type": "integer"
      },
      "available": {
        "type": "boolean"
      }
    },
    "required": [
      "age",
      "available"
    ]
  }
}'
```

##### Response

```json
{
  "model": "llama3.1:8b",
  "created_at": "2024-12-06T00:48:09.983619Z",
  "response": "{\n  \"age\": 22,\n  \"available\": true\n}",
  "done": true,
  "done_reason": "stop",
  "context": [1, 2, 3],
  "total_duration": 1075509083,
  "load_duration": 567678166,
  "prompt_eval_count": 28,
  "prompt_eval_duration": 236000000,
  "eval_count": 16,
  "eval_duratio

# Эмбеддинги

In [25]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

Загружаем модель эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches: 100%|██████████| 16/16 [00:00<00:00, 34.32it/s]

Размер матрицы эмбеддингов: (500, 384)


In [26]:
print(f"Первый чанк: {chunks[0]['text'][:80]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

Первый чанк: # Documentation

### Getting Started
* [Quickstart](https://docs.ollama.com/quic...
Его эмбеддинг (первые 10 чисел): [-0.06130742 -0.05510944 -0.03115823 -0.0100545   0.03433185  0.04836936
 -0.08265638  0.02022991 -0.00097116  0.04949648]
Длина вектора (норма): 1.0000


### Строим векторный индекс

In [27]:
import faiss

dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"В индексе {index.ntotal} векторов размерности {dim}")

В индексе 500 векторов размерности 384


### Первый поиск

In [29]:
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    """Возвращает top_k чанков, наиболее близких к запросу по смыслу."""
    # 1. Считаем эмбеддинг запроса
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)

    # 2. Ищем ближайших соседей
    scores, indices = index.search(query_emb, top_k)

    # 3. Собираем результат с метаданными
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

In [30]:
question = "Как сменить папку, в которой Ollama хранит модели?"
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

Запрос: Как сменить папку, в которой Ollama хранит модели?

#1  [docs/windows.mdx]  score=0.765
To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `OLLAMA_MODELS` in your user account.

1. Start the Settings (Windows 11) or Cont ...

#2  [docs/macos.mdx]  score=0.629
## Filesystem Requirements

The preferred method of installation is to mount the `ollama.dmg` and drag-and-drop the Ollama application to the system-wide `Applications` folder.  Upon startup, the Olla ...

#3  [docs/faq.mdx]  score=0.620
Alternatively, you can change the amount of time all models are loaded into memory by setting the `OLLAMA_KEEP_ALIVE` environment variable when starting the Ollama server. The `OLLAMA_KEEP_ALIVE` vari ...



In [34]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Загружаем модель эмбеддингов
embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. Считаем эмбеддинги для всех чанков
texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

# 3. Строим FAISS-индекс
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"В индексе {index.ntotal} векторов размерности {dim}")

# 4. Функция поиска
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)
    
    scores, indices = index.search(query_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

# 5. Проверяем на трёх вопросах
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    for r in vector_search(q, top_k=3):
        print(f"  [{r['source']}]  score={r['score']:.3f}")
        print(f"  {r['text'][:120].strip()}...\n")

Batches: 100%|██████████| 16/16 [00:00<00:00, 32.53it/s]


В индексе 500 векторов размерности 384
=== What HTTP method does /api/generate use? ===
  [docs/api/introduction.mdx]  score=0.472
  ## Base URL

After installation, Ollama's API is served by default at:

```
http://localhost:11434/api
```

For running...

  [docs/faq.mdx]  score=0.461
  ```shell
docker build -t ollama-with-ca .
docker run -d -e HTTPS_PROXY=https://my.proxy.example.com -p 11434:11434 ollam...

  [docs/api/introduction.mdx]  score=0.442
  ## Example request

Once Ollama is running, its API is automatically available and can be accessed via `curl`:

```shell...

=== Как сменить папку, где хранятся модели? ===
  [docs/faq.mdx]  score=0.586
  ## Where are models stored?

- macOS: `~/.ollama/models`
- Linux: `/usr/share/ollama/.ollama/models`
- Windows: `C:\User...

  [docs/windows.mdx]  score=0.581
  To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `...

  [docs/modelfile.mdx]  score=0.440
  ```shell
oll

# Делаем первый запрос к модели

```bash
ollama list
ollama serve
ollama run huihui_ai/Qwen3.8-abliterated:latest
```


In [47]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen2.5:1.5b", #"huihui_ai/Qwen3.8-abliterated:latest", # 
        "prompt": "Привет! Назови одно интересное свойство Python в одном предложении.",
        "stream": False,
    },
    timeout=120,
)

data = response.json()
print(data["response"])

Python является низкоуровневым языком с небольшим количеством специфических тонкостей, что делает его мощным инструментом для оптимизации и ветвления кода.


In [53]:
import requests

OLLAMA_URL = "http://localhost:11434"
LLM_MODEL = "qwen2.5:1.5b"


def generate_answer(prompt: str) -> str:
    """Отправляет промпт в Ollama и возвращает сгенерированный ответ."""
    response = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={
        "model": "qwen2.5:1.5b", #"huihui_ai/Qwen3.8-abliterated:latest", # 
        "prompt": prompt,
        "stream": False,
    },
    timeout=120,
)
    response.raise_for_status()
    data = response.json()
    return data["response"]


# Пробуем
answer = generate_answer("Что такое RAG в одном предложении?")
print(answer)

RAG stands for "Retrieval-Augmented Generation". It's a type of machine learning technique where the generation of a text is enhanced by a set of previously generated texts, often from a large knowledge base or database. Here's a brief explanation:

1. "Retrieval" - This refers to the process of searching for relevant documents or information to use as a basis for generating new text.

2. "Augmented" - This means that the generated text incorporates elements from these retrieved texts, typically in a way that makes the output more coherent, relevant, and contextually appropriate.

RAG works by:

1. Preparing a large knowledge base of documents or a text corpus.

2. Using a search engine to find relevant documents.

3. Generating text that includes segments or entire passages from the documents.

4. Optionally, combining this text with pre-generated similar outputs or training data.

The key idea is that the generated text can benefit from the existing knowledge, improving its quality a

In [55]:
PROMPT_TEMPLATE = """Ты ассистент по документации Ollama.
Отвечай на вопрос пользователя, опираясь только на приведённые ниже фрагменты документации.
Если ответа в фрагментах нет  честно скажи, что не знаешь.
Ответ давай на русском языке.
В конце ответа на отдельной строке укажи источник в формате: Источник: <имя файла>.

Фрагменты документации:
{context}

Вопрос пользователя: {question}

Ответ:"""


def build_prompt(question: str, retrieved: list[dict]) -> str:
    """Собирает промпт из системной инструкции, контекста и вопроса."""
    context_parts = []
    for r in retrieved:
        context_parts.append(f"[файл: {r['source']}]\n{r['text']}")

    context = "\n\n---\n\n".join(context_parts)
    return PROMPT_TEMPLATE.format(context = context, question=question) 

# Тестовый прогон
question = "Как сменить папку, в которой Ollama хранит модели?"

retrieved = vector_search(question, top_k=3)
prompt = build_prompt(question, retrieved)

print("Промпт:")
print(prompt[:500], "...\n")

print("Ответ модели:")
answer = generate_answer(prompt)
print(answer)

Промпт:
Ты ассистент по документации Ollama.
Отвечай на вопрос пользователя, опираясь только на приведённые ниже фрагменты документации.
Если ответа в фрагментах нет  честно скажи, что не знаешь.
Ответ давай на русском языке.
В конце ответа на отдельной строке укажи источник в формате: Источник: <имя файла>.

Фрагменты документации:
[файл: docs/windows.mdx]
To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `OLLAMA_MODELS` in your user  ...

Ответ модели:
Чтобы изменить папку, в которой Ollama хранит модели, следует:

1. Откройте Settings (в Windows 11) или Control Panel (в Windows 10).
2. Найдите и выберите элемент "Environment Variables".
3. В разделе "Edit environment variables for your account" выберите переменную окружения `OLLAMA_MODELS`.
4. Здесь можно изменить или создать новую директорию для хранения моделей.

Помните, что, если Ollama уже запущен, нужно выключить трей-приложение и запустить его снова из Start

# Сборка и тестирование RAG-пайплайна

In [61]:
def ask_assistant(question: str, top_k: int = 3) -> str:
    """Полный RAG-пайплайн: поиск, сборка промпта, генерация ответа."""
    retrieved = vector_search(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)
    answer = generate_answer(prompt)
    return answer

# Первый сквозной запуск
question = "Какой HTTP-метод использует эндпоинт /api/generate?"
print(f"Вопрос: {question}\n")
print(f"Ответ:\n{ask_assistant(question)}")

Вопрос: Какой HTTP-метод использует эндпоинт /api/generate?

Ответ:
HTTP-метод использует метод POST.

Источник: docs/api/generate.mdx

Обратите внимание, что в оригинальном вопросе не было указана версия документации, поэтому исходный URL не приведён.


### Тестируем на разных вопросах

In [62]:
def debug_ask(question: str, top_k: int = 3):
    """Показывает и найденные чанки, и финальный ответ - удобно для отладки."""
    print(f"Вопрос: {question} \n")

    retrieved = vector_search(question, top_k=top_k)
    print("Найденные чанки:")
    for i, r in enumerate(retrieved, 1):
        print(f"  #{i} [{r['source']}] score={r['score']:.3f}")
        print(f"     {r['text'][:120].strip()}...")
    print()

    prompt = build_prompt(question, retrieved)
    answer = generate_answer(prompt)
    print(f"Ответ модели:\n{answer}\n")

In [63]:
test_questions = [
    "Какой HTTP-метод использует эндпоинт /api/generate?",
    "Что такое Modelfile?",
    "Как удалить локально скачанную модель через CLI?",
    "Какая текущая цена подписки на Ollama Cloud?",
]

for q in test_questions:
    debug_ask(q)
    print("-" * 70)
    print()

Вопрос: Какой HTTP-метод использует эндпоинт /api/generate? 

Найденные чанки:
  #1 [docs/faq.mdx] score=0.583
     ```shell
docker build -t ollama-with-ca .
docker run -d -e HTTPS_PROXY=https://my.proxy.example.com -p 11434:11434 ollam...
  #2 [docs/api/introduction.mdx] score=0.498
     ## Base URL

After installation, Ollama's API is served by default at:

```
http://localhost:11434/api
```

For running...
  #3 [docs/api/anthropic-compatibility.mdx] score=0.497
     ```shell
curl http://localhost:11434/v1/messages \
    -H "Content-Type: application/json" \
    -d '{
        "model":...

Ответ модели:
HTTP-метод использует POST.

Источник: docs/api/generate.mdx

----------------------------------------------------------------------

Вопрос: Что такое Modelfile? 

Найденные чанки:
  #1 [docs/modelfile.mdx] score=0.715
     ---
title: Modelfile Reference
---

A Modelfile is the blueprint to create and share customized models using Ollama....
  #2 [docs/import.mdx] score=0.601
     - a